In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import col, from_json, explode_outer

company_profile_schema = ArrayType(StructType([
    StructField("symbol", StringType()),
    StructField("price", DoubleType()),
    StructField("marketCap", LongType()),
    StructField("beta", DoubleType()),
    StructField("lastDividend", DoubleType()),
    StructField("range", StringType()),
    StructField("change", DoubleType()),
    StructField("changePercentage", DoubleType()),
    StructField("volume", LongType()),
    StructField("averageVolume", LongType()),
    StructField("companyName", StringType()),
    StructField("currency", StringType()),
    StructField("cik", StringType()),
    StructField("isin", StringType()),
    StructField("cusip", StringType()),
    StructField("exchangeFullName", StringType()),
    StructField("exchange", StringType()),
    StructField("industry", StringType()),
    StructField("website", StringType()),
    StructField("description", StringType()),
    StructField("ceo", StringType()),
    StructField("sector", StringType()),
    StructField("country", StringType()),
    StructField("fullTimeEmployees", StringType()),
    StructField("phone", StringType()),
    StructField("address", StringType()),
    StructField("city", StringType()),
    StructField("state", StringType()),
    StructField("zip", StringType()),
    StructField("image", StringType()),
    StructField("ipoDate", StringType()),
    StructField("defaultImage", BooleanType()),
    StructField("isEtf", BooleanType()),
    StructField("isActivelyTrading", BooleanType()),
    StructField("isAdr", BooleanType()),
    StructField("isFund", BooleanType())
]))

bronze_df = spark.read.table("workspace.stock_data.company_profiles_bronze")

parsed_df = bronze_df.withColumn(
    "parsed",
    from_json(col("raw_json"), company_profile_schema)
)

exploded_df = (
    parsed_df
        .withColumn("profile", explode_outer("parsed"))
        .filter((col('profile.isEtf') == False) & (col('profile.isFund') == False) & (col('profile.isAdr') == False) & (col('profile.isActivelyTrading') == True))
)

silver_df = exploded_df.select(
    col("profile.symbol").alias("symbol"),
    col("profile.companyName").alias("company_name"),
    col("profile.sector"),
    col("profile.industry"),
    col("profile.description"),
    col("profile.exchange"),
    col("profile.country"),
    col("profile.price"),
    col("profile.ipoDate").alias("ipo_date"),
    col("profile.marketCap").alias("market_cap"),
    col("profile.currency"),
    col("ingest_timestamp")
)

cleaned_silver_df = (
    silver_df
        .dropDuplicates(["symbol"])
        .filter(col("symbol").isNotNull())
)

cleaned_silver_df.write.format("delta").mode("overwrite").saveAsTable('workspace.stock_data.company_profiles_silver')

In [0]:
%sql
SELECT *
FROM workspace.stock_data.company_profiles_silver